In [1]:
#find only DE 14 without overlap genes for EP, EC amd PC#

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import binomtest, spearmanr, fisher_exact, hypergeom, chi2
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

In [3]:

CELLTYPES = ["Monocytes","NK","pDCs","ASCs","B_cells","CD2_neg_T","CD2_pos_T","CD4_Tcells","CD8T_cells"]
CONTRASTS = {"EP": "ext_per", "EC": "ext_con", "PC": "per_con"}

BASE14 = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/Project_Fang_10X/Project_Fang_10X/MAST_DE_14dpi")
BASE84 = Path("/work/ABG/mkapoor/mkapoor/PRRSV/PRRSV_cellranger_v97/MAST_DE_84dpi")
OUTROOT = Path("/work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84")
OUTROOT.mkdir(parents=True, exist_ok=True)

FDR_CUTOFF = 0.05
LFC_CUTOFF = 0.05
COLS = {"gene": "primerid", "fdr": "fdr", "logfc": "logFC"}  # preferred names

def load_and_standardize(fn, cols):
    df = pd.read_csv(fn)
    def pick(cands):
        low = {c.lower(): c for c in df.columns}
        for c in cands:
            if c.lower() in low:
                return low[c.lower()]
        raise KeyError(f"None of {cands} in {list(df.columns)}")

    gene_col = pick([cols["gene"], "gene", "primerid", "symbol"])
    fdr_col  = pick([cols["fdr"], "fdr", "adj.P.Val", "padj", "qvalue"])
    lfc_col  = pick([cols["logfc"], "logfc", "log2fc", "avg_log2FC", "coef"])

    df = df[[gene_col, fdr_col, lfc_col]].rename(columns={gene_col:"gene", fdr_col:"fdr", lfc_col:"logFC"})
    # coerce numeric, drop bad rows
    df["fdr"] = pd.to_numeric(df["fdr"], errors="coerce")
    df["logFC"] = pd.to_numeric(df["logFC"], errors="coerce")
    df = df.dropna(subset=["gene", "fdr", "logFC"])

    # if duplicates, keep the row with min fdr; tie-break by largest |logFC|
    df = (df.sort_values(["gene","fdr", "logFC"], ascending=[True, True, False])
            .groupby("gene", as_index=False).first())

    cond_up   = (df["fdr"] < FDR_CUTOFF) & (df["logFC"] >  LFC_CUTOFF)
    cond_down = (df["fdr"] < FDR_CUTOFF) & (df["logFC"] < -LFC_CUTOFF)
    df["direction"] = np.where(cond_up, "Upregulated",
                        np.where(cond_down, "Downregulated", "Non-significant"))
    return df

def read_files(base_dir, celltype, frag):
    ct_dir = base_dir / celltype
    candidates = [ct_dir / f"Mast_{celltype}_{frag}.csv", ct_dir / f"MAST_{celltype}_{frag}.csv"]
    for p in candidates:
        if p.exists():
            return p
    if ct_dir.exists():
        for p in ct_dir.glob("*.csv"):
            if frag in p.name:
                return p
    raise FileNotFoundError(f"No MAST file for {celltype} + '{frag}' in {ct_dir}")

def analyze(celltype, contrast_key):
    frag = CONTRASTS[contrast_key]
    fn14 = read_files(BASE14, celltype, frag)
    fn84 = read_files(BASE84, celltype, frag)

    d14 = load_and_standardize(fn14, COLS)
    d84 = load_and_standardize(fn84, COLS)

    # significant sets (by timepoint)
    sig14 = set(d14.loc[(d14["fdr"] < FDR_CUTOFF) & (d14["logFC"].abs() > LFC_CUTOFF), "gene"])

    sig84 = set(d84.loc[(d84["fdr"] < FDR_CUTOFF) & (d84["logFC"].abs() > LFC_CUTOFF), "gene"])

    unique14 = sig14 - sig84         # only at 14
    unique84 = sig84 - sig14         # only at 84
    shared   = sig14 & sig84         # in both

    def table_from_ids(df, ids, tag):
        sub = df[df["gene"].isin(ids)].copy()
        sub["timepoint"] = tag
        return sub.sort_values(["direction","logFC"], ascending=[True, False])

    unique14_df = table_from_ids(d14, unique14, "14dpi")
    unique84_df = table_from_ids(d84, unique84, "84dpi")
    shared14_df = table_from_ids(d14, shared,   "14dpi")
    shared84_df = table_from_ids(d84, shared,   "84dpi")

    outdir = OUTROOT / contrast_key / celltype
    outdir.mkdir(parents=True, exist_ok=True)

    unique14_df.to_csv(outdir / "unique_sig_14dpi.csv", index=False)
    unique84_df.to_csv(outdir / "unique_sig_84dpi.csv", index=False)
    # optional: write shared for reference
    pd.concat([shared14_df, shared84_df], ignore_index=True).to_csv(outdir / "shared_sig_both.csv", index=False)

    return {
        "celltype": celltype,
        "contrast": contrast_key,
        "n_sig_14": len(sig14),
        "n_sig_84": len(sig84),
        "n_unique_14": len(unique14),
        "n_unique_84": len(unique84),
        "n_shared": len(shared),
    }

# main
all_rows = []
for contrast in CONTRASTS:
    for ct in CELLTYPES:
        try:
            all_rows.append(analyze(ct, contrast))
        except FileNotFoundError as e:
            print(f"[WARN] {e}")
        except Exception as e:
            print(f"[ERROR] {ct} {contrast}: {e}")

#  write a summary CSV
if all_rows:
    pd.DataFrame(all_rows).to_csv(OUTROOT / "summary_unique_at_14_or_84_all.csv", index=False)

print("Done. Summaries saved under:", OUTROOT)

Done. Summaries saved under: /work/ABG/mkapoor/mkapoor/Project_Fang_10X/14dpi_PRRSV/compare_14_84


In [4]:
##GO ANALYSIS #
#in R